# Final GPT-2 Small Activation-Steering Test Generation

This notebook creates a paired test set for three intervention protocols—`classic`,
`random`, and contextual `gate`—using one normalized SAE decoder direction. It only
generates and validates JSONL results; text scoring and Pareto analysis are intentionally
outside this test-generation stage.

The test prompts are held out: the alpha grid, random-gate probability, and content-gate
threshold must be fixed before this notebook is run and must not be tuned on these prompts.

## 1. Imports and environment

The notebook records key package versions and selects CUDA when available, otherwise CPU.
No function assumes a hard-coded CUDA device.

In [15]:
import inspect
import json
import os
import random
import time
import warnings
from dataclasses import dataclass, field
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd
import torch
import transformer_lens
import sae_lens
from sae_lens import SAE
from transformer_lens import HookedTransformer


def package_version(package_name: str) -> str:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "unknown"


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"torch: {torch.__version__}")
print(f"transformer_lens: {package_version('transformer-lens')}")
print(f"sae_lens: {package_version('sae-lens')}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Device: cuda
torch: 2.11.0+cu128
transformer_lens: 4.0.0b1
sae_lens: 6.49.0
numpy: 1.26.4
pandas: 2.3.3
CUDA device: NVIDIA GeForce RTX 5060


## 2. Experiment configuration

All parameters intended for modification are declared here. Keep the test set held out:
do not select `ALPHAS`, `CONTENT_GATE_THRESHOLD`, or `RANDOM_GATE_PROBABILITY` by looking
at generations from `TEST_PROMPTS_PATH`.

In [16]:
MODEL_NAME = "gpt2-small"
SAE_RELEASE = "gpt2-small-res-jb"
SAE_ID = "blocks.6.hook_resid_pre"
HOOK_NAME = "blocks.6.hook_resid_pre"

TEST_PROMPTS_PATH = "data/chatgpt.json"
TEST_PROMPTS_KEY = "test"  # Set to None when the JSON root is directly a list.

OUTPUT_ROOT = "results/test"
OVERWRITE_RESULTS = False

CONCEPT_NAME = "violence"
CONCEPT_FEATURE_ID = 20026

ALPHAS = [0.0, 2.5, 5.0, 7.5, 10.0, 12.5, 15.0, 17.5, 20.0, 22.5, 25.0, 27.5, 30.0, 32.5, 35.0, 37.5, 40.0, 42.5, 45.0, 47.5, 50.0]
SEEDS = [42, 43, 44, 45, 46]

TEMPERATURE = 0.9
TOP_P = 0.9
TOP_K = 50
MAX_NEW_TOKENS = 100

RANDOM_GATE_PROBABILITY = 0.4
CONTENT_GATE_THRESHOLD = 6.536377

BASE_GATE_SEED = 10000
VECTOR_NORM_EPS = 1e-8

METHODS = ["classic", "random", "gate"]

## 3. Model and SAE loading

GPT-2 small and the residual-stream SAE are loaded onto the selected device. Both are put
into evaluation mode. The SAE loader compatibility helper supports common SAE Lens releases
that return either an SAE directly or a tuple whose first element is the SAE.

In [17]:
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()


def load_sae_compatibly(release: str, sae_id: str, device: torch.device) -> SAE:
    loaded = SAE.from_pretrained(release=release, sae_id=sae_id, device=str(device))
    loaded_sae = loaded[0] if isinstance(loaded, tuple) else loaded
    if not isinstance(loaded_sae, SAE):
        raise TypeError(f"Expected SAE, received {type(loaded_sae)!r}")
    return loaded_sae


sae = load_sae_compatibly(SAE_RELEASE, SAE_ID, DEVICE)
sae.eval()

assert HOOK_NAME == SAE_ID, "This experiment expects the SAE and hook point to match."
assert sae.W_dec.shape[-1] == model.cfg.d_model
print(f"Loaded model: {MODEL_NAME}")
print(f"Loaded SAE: {SAE_RELEASE} / {SAE_ID}")
print(f"Residual width: {model.cfg.d_model}; SAE features: {sae.W_dec.shape[0]}")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8249.70it/s]


Loaded pretrained model gpt2-small into HookedTransformer
Loaded model: gpt2-small
Loaded SAE: gpt2-small-res-jb / blocks.6.hook_resid_pre
Residual width: 768; SAE features: 24576


C:\Users\supervisor.LTCPC-280426-01\tlab\lib\site-packages\sae_lens\saes\sae.py:254: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


## 4. Test prompts loading

Two JSON layouts are accepted: a list at the root, or a dictionary containing the list under
`TEST_PROMPTS_KEY`. Validation rejects missing files, empty lists, and blank/non-string items.

In [18]:
def load_test_prompts(path: str, key: Optional[str]) -> List[str]:
    prompt_path = Path(path)
    if not prompt_path.is_file():
        raise FileNotFoundError(
            f"Test prompts file not found: {prompt_path}. "
            "Create it before running the full notebook."
        )

    with prompt_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    if key is None:
        prompts = payload
    else:
        if not isinstance(payload, dict):
            raise TypeError(
                f"TEST_PROMPTS_KEY={key!r}, so the JSON root must be an object."
            )
        if key not in payload:
            raise KeyError(f"Key {key!r} is missing from {prompt_path}.")
        prompts = payload[key]

    if not isinstance(prompts, list):
        raise TypeError("Loaded prompts must be a list.")
    if not prompts:
        raise ValueError("The test prompt list is empty.")
    invalid_indices = [
        index for index, prompt in enumerate(prompts)
        if not isinstance(prompt, str) or not prompt.strip()
    ]
    if invalid_indices:
        raise ValueError(
            "Every prompt must be a non-empty string. Invalid indices: "
            f"{invalid_indices[:10]}"
        )
    return prompts


test_prompts = load_test_prompts(TEST_PROMPTS_PATH, TEST_PROMPTS_KEY)
print(f"Loaded {len(test_prompts)} test prompts.")
for index, example in enumerate(test_prompts[:3]):
    print(f"[{index}] {example}")

Loaded 20 test prompts.
[0] After lunch, Maya returned to her desk and found a handwritten note beside the keyboard.
[1] The train stopped outside a small town just as the sun disappeared behind the hills.
[2] Daniel spent the afternoon sorting old photographs that had been stored in a wooden box.


## 5. Steering-vector preparation

The decoder row of the selected SAE feature defines the raw direction $v$. Every method uses
exactly the same unit vector:

$$\hat v=\frac{v}{\max(\lVert v\rVert_2,\varepsilon)}.$$

At autoregressive step $t$, only the final residual position is changed:

$$h'_t=h_t+\alpha g_t\hat v.$$

All earlier sequence positions remain untouched.

In [19]:
if not (0 <= CONCEPT_FEATURE_ID < sae.W_dec.shape[0]):
    raise IndexError(
        f"CONCEPT_FEATURE_ID={CONCEPT_FEATURE_ID} is outside "
        f"[0, {sae.W_dec.shape[0] - 1}]."
    )

raw_steering_vector = sae.W_dec[CONCEPT_FEATURE_ID].detach()
raw_vector_norm = raw_steering_vector.norm()
steering_vector = raw_steering_vector / raw_vector_norm.clamp_min(VECTOR_NORM_EPS)

assert torch.isfinite(steering_vector).all()
assert torch.allclose(
    steering_vector.norm(),
    torch.tensor(1.0, device=steering_vector.device),
    atol=1e-5,
)

raw_vector_norm_value = float(raw_vector_norm.detach().cpu())
normalized_vector_norm_value = float(steering_vector.norm().detach().cpu())

print(f"CONCEPT_FEATURE_ID: {CONCEPT_FEATURE_ID}")
print(f"Vector shape: {tuple(steering_vector.shape)}")
print(f"Raw vector norm: {raw_vector_norm_value:.8f}")
print(f"Normalized vector norm: {normalized_vector_norm_value:.8f}")

CONCEPT_FEATURE_ID: 20026
Vector shape: (768,)
Raw vector norm: 1.00004470
Normalized vector norm: 1.00000000


## 6. Content-gate preparation

GPT-2 vocabulary items that decode as a whole alphabetic word with a leading space are split
into function words and other (content-candidate) words. The unembedding contrast is

$$d_{content}=\operatorname{mean}(W_U[:,C])-\operatorname{mean}(W_U[:,F]),$$

and each SAE decoder feature receives score $s_j=W_{dec,j}\cdot d_{content}$. During generation,
the gate score is the SAE-activation-weighted sum of these feature scores. The threshold is
fixed in configuration and is not calibrated on test prompts.

In [20]:
FUNCTION_WORDS: Set[str] = {
    # Articles and determiners
    "a", "an", "the", "this", "that", "these", "those", "some", "any", "each",
    "every", "either", "neither", "much", "many", "few", "fewer", "little", "less",
    "all", "both", "another", "other", "such", "what", "whatever", "which", "whichever",
    # Pronouns
    "i", "me", "my", "mine", "myself", "you", "your", "yours", "yourself", "yourselves",
    "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself",
    "we", "us", "our", "ours", "ourselves", "they", "them", "their", "theirs", "themselves",
    "who", "whom", "whose", "whoever", "whomever", "one", "ones", "someone", "anyone",
    "everyone", "nobody", "nothing", "something", "anything", "everything",
    # Prepositions
    "about", "above", "across", "after", "against", "along", "among", "around", "as", "at",
    "before", "behind", "below", "beneath", "beside", "between", "beyond", "by", "despite",
    "down", "during", "except", "for", "from", "in", "inside", "into", "like", "near", "of",
    "off", "on", "onto", "out", "outside", "over", "past", "per", "since", "through",
    "throughout", "to", "toward", "towards", "under", "underneath", "until", "up", "upon",
    "via", "with", "within", "without",
    # Conjunctions
    "and", "but", "or", "nor", "for", "yet", "so", "although", "because", "if", "unless",
    "while", "whereas", "whether", "than", "though", "once", "when", "whenever", "where",
    "wherever", "before", "after", "until", "since",
    # Auxiliary and modal verbs
    "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having",
    "do", "does", "did", "doing", "can", "could", "may", "might", "must", "shall", "should",
    "will", "would", "ought", "need", "dare",
    # Negations and common grammatical adverbs
    "no", "not", "never", "neither", "nor", "hardly", "scarcely", "barely", "only", "also",
    "even", "just", "still", "already", "yet", "too", "very", "quite", "rather", "then",
    "there", "here", "now", "again", "ever", "always", "often", "sometimes", "usually",
}


def build_content_and_function_token_ids(
    tokenizer: Any,
    function_words: Set[str],
) -> Tuple[torch.Tensor, torch.Tensor]:
    content_token_ids: List[int] = []
    function_token_ids: List[int] = []

    for token_id in range(len(tokenizer)):
        decoded = tokenizer.decode([token_id])
        # A leading literal space marks a complete GPT-2 word token for this protocol.
        if not decoded.startswith(" "):
            continue
        word = decoded.strip()
        if not word or not word.isalpha():
            continue
        if word.lower() in function_words:
            function_token_ids.append(token_id)
        else:
            content_token_ids.append(token_id)

    if not content_token_ids or not function_token_ids:
        raise ValueError("Failed to construct non-empty content/function token sets.")

    return (
        torch.tensor(content_token_ids, device=DEVICE, dtype=torch.long),
        torch.tensor(function_token_ids, device=DEVICE, dtype=torch.long),
    )


content_ids, function_ids = build_content_and_function_token_ids(
    model.tokenizer,
    FUNCTION_WORDS,
)

content_direction = model.W_U[:, content_ids].mean(dim=-1)
function_direction = model.W_U[:, function_ids].mean(dim=-1)
content_contrast_direction = content_direction - function_direction
feature_content_scores = (sae.W_dec @ content_contrast_direction).detach()

assert feature_content_scores.shape == (sae.W_dec.shape[0],)
assert torch.isfinite(feature_content_scores).all()
print(f"Whole-word content token IDs: {content_ids.numel()}")
print(f"Whole-word function token IDs: {function_ids.numel()}")
print(f"Feature content scores shape: {tuple(feature_content_scores.shape)}")

Whole-word content token IDs: 31632
Whole-word function token IDs: 472
Feature content scores shape: (24576,)


## 7. Hook implementations

`classic` uses $g_t=1$, `random` draws $g_t\sim\mathrm{Bernoulli}(q)$ from a dedicated
generator, and `gate` uses the fixed content-score threshold. Every hook clones its input and
writes only `tensor[:, -1:, :]`. Diagnostics remain as detached device tensors throughout the
autoregressive loop and are transferred to CPU only once after generation.

In [21]:
@dataclass
class GateDiagnostics:
    gate_tensors: List[torch.Tensor] = field(default_factory=list)
    content_score_tensors: List[torch.Tensor] = field(default_factory=list)

    def add(self, gate: torch.Tensor, content_score: Optional[torch.Tensor] = None) -> None:
        self.gate_tensors.append(gate.detach())
        if content_score is not None:
            self.content_score_tensors.append(content_score.detach())

    def summarize(self) -> Dict[str, Optional[float]]:
        if self.gate_tensors:
            gates = torch.cat([tensor.reshape(-1) for tensor in self.gate_tensors])
            gates_cpu = gates.to(device="cpu", dtype=torch.float32)
            gate_calls = int(gates_cpu.numel())
            gate_triggered = int(gates_cpu.sum().item())
            gate_trigger_rate = float(gates_cpu.mean().item())
        else:
            gate_calls = 0
            gate_triggered = 0
            gate_trigger_rate = 0.0

        score_min: Optional[float] = None
        score_mean: Optional[float] = None
        score_max: Optional[float] = None
        if self.content_score_tensors:
            scores = torch.cat(
                [tensor.reshape(-1) for tensor in self.content_score_tensors]
            ).to(device="cpu", dtype=torch.float32)
            score_min = float(scores.min().item())
            score_mean = float(scores.mean().item())
            score_max = float(scores.max().item())

        return {
            "gate_calls": gate_calls,
            "gate_triggered": gate_triggered,
            "gate_trigger_rate": gate_trigger_rate,
            "content_score_min": score_min,
            "content_score_mean": score_mean,
            "content_score_max": score_max,
        }


HookFunction = Callable[[torch.Tensor, Any], torch.Tensor]


def make_steering_hook(
    method: str,
    alpha: float,
    unit_steering_vector: torch.Tensor,
    diagnostics: GateDiagnostics,
    *,
    gate_generator: Optional[torch.Generator] = None,
    random_gate_probability: Optional[float] = None,
    loaded_sae: Optional[SAE] = None,
    loaded_feature_content_scores: Optional[torch.Tensor] = None,
    content_gate_threshold: Optional[float] = None,
) -> HookFunction:
    if method not in METHODS:
        raise ValueError(f"Unknown steering method: {method!r}")
    if method == "random" and (gate_generator is None or random_gate_probability is None):
        raise ValueError("Random steering requires its own generator and gate probability.")
    if method == "gate" and (
        loaded_sae is None
        or loaded_feature_content_scores is None
        or content_gate_threshold is None
    ):
        raise ValueError("Content-gated steering requires SAE scores and a threshold.")

    def steering_hook(tensor: torch.Tensor, hook: Any) -> torch.Tensor:
        del hook
        output = tensor.clone()
        last_tensor = tensor[:, -1:, :]
        vector = unit_steering_vector.to(device=tensor.device, dtype=tensor.dtype).reshape(1, 1, -1)

        if method == "classic":
            gate = torch.ones(
                (*last_tensor.shape[:-1], 1),
                device=tensor.device,
                dtype=torch.bool,
            )
            steered_last = last_tensor + float(alpha) * vector
            diagnostics.add(gate)

        elif method == "random":
            gate = torch.rand(
                (*last_tensor.shape[:-1], 1),
                device=tensor.device,
                generator=gate_generator,
            ) < float(random_gate_probability)
            steered_last = (
                last_tensor
                + gate.to(last_tensor.dtype) * float(alpha) * vector
            )
            diagnostics.add(gate)

        else:
            sae_acts = loaded_sae.encode(last_tensor)
            scores = loaded_feature_content_scores.to(
                device=sae_acts.device,
                dtype=sae_acts.dtype,
            )
            content_score = (
                sae_acts * scores.reshape(1, 1, -1)
            ).sum(dim=-1, keepdim=True)
            gate = content_score >= float(content_gate_threshold)
            steered_last = (
                last_tensor
                + gate.to(last_tensor.dtype) * float(alpha) * vector
            )
            diagnostics.add(gate, content_score)

        output[:, -1:, :] = steered_last
        return output

    return steering_hook

## 8. Reproducible generation utilities

Token sampling uses the global PyTorch RNG after `set_generation_seed`. Random gating uses a
separate device-local `torch.Generator`, seeded deterministically without Python's randomized
`hash`. Hooks are removed in `finally`, including when generation raises an exception.

In [22]:
def set_generation_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_gate_seed(
    generation_seed: int,
    prompt_index: int,
    alpha_index: int,
) -> int:
    return int(
        BASE_GATE_SEED
        + generation_seed * 100_000
        + prompt_index * 1_000
        + alpha_index
    )


GENERATION_KWARGS: Dict[str, Any] = {
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "do_sample": True,
    "verbose": False,
}
if "use_past_kv_cache" in inspect.signature(model.generate).parameters:
    GENERATION_KWARGS["use_past_kv_cache"] = True
else:
    warnings.warn("This TransformerLens version does not expose use_past_kv_cache.")


def generate_text(prompt: str) -> str:
    generated = model.generate(prompt, **GENERATION_KWARGS)
    if isinstance(generated, str):
        return generated
    if torch.is_tensor(generated):
        token_ids = generated[0] if generated.ndim == 2 else generated
        return model.to_string(token_ids)
    raise TypeError(f"Unexpected model.generate output type: {type(generated)!r}")


def generate_with_method(
    prompt: str,
    method: str,
    alpha: float,
    gate_seed: Optional[int],
) -> Tuple[str, Dict[str, Optional[float]]]:
    diagnostics = GateDiagnostics()
    gate_generator: Optional[torch.Generator] = None
    if method == "random":
        if gate_seed is None:
            raise ValueError("gate_seed is required for random steering.")
        gate_generator = torch.Generator(device=DEVICE)
        gate_generator.manual_seed(int(gate_seed))

    hook_fn = make_steering_hook(
        method=method,
        alpha=float(alpha),
        unit_steering_vector=steering_vector,
        diagnostics=diagnostics,
        gate_generator=gate_generator,
        random_gate_probability=RANDOM_GATE_PROBABILITY if method == "random" else None,
        loaded_sae=sae if method == "gate" else None,
        loaded_feature_content_scores=feature_content_scores if method == "gate" else None,
        content_gate_threshold=CONTENT_GATE_THRESHOLD if method == "gate" else None,
    )

    model.add_hook(HOOK_NAME, hook_fn, dir="fwd")
    try:
        generated_text = generate_text(prompt)
    finally:
        # Guaranteed cleanup prevents hooks leaking into baseline or later generations.
        model.reset_hooks()

    return generated_text, diagnostics.summarize()

## 9. Test generation

Loop order is prompt → generation seed → alpha → method. For each `(prompt, seed)`, the
no-steering text is generated once and reused in every record. Immediately before every
steered generation the same sampling seed is restored, establishing paired comparisons.
Output files are preflight-checked before any generation begins.

In [ ]:
output_paths: Dict[str, Path] = {
    method: Path(OUTPUT_ROOT) / method / "results.jsonl"
    for method in METHODS
}

for path in output_paths.values():
    os.makedirs(path.parent, exist_ok=True)

existing_paths = [path for path in output_paths.values() if path.exists()]
if existing_paths and not OVERWRITE_RESULTS:
    formatted_paths = "\n".join(f"  - {path}" for path in existing_paths)
    raise FileExistsError(
        "Result files already exist and OVERWRITE_RESULTS=False. "
        "Refusing to mix experiments. Rename/remove them or explicitly set "
        f"OVERWRITE_RESULTS=True:\n{formatted_paths}"
    )

expected_records_per_method = len(test_prompts) * len(SEEDS) * len(ALPHAS)
alpha_zero_mismatches: List[Tuple[str, int, int]] = []
experiment_started_at = time.perf_counter()
writers: Dict[str, Any] = {}

try:
    writers = {
        method: output_paths[method].open("w", encoding="utf-8")
        for method in METHODS
    }

    with torch.inference_mode():
        for prompt_index, prompt in enumerate(test_prompts):
            for generation_seed in SEEDS:
                set_generation_seed(generation_seed)
                no_steering = generate_text(prompt)

                for alpha_index, alpha in enumerate(ALPHAS):
                    for method in METHODS:
                        method_gate_seed = (
                            compute_gate_seed(
                                generation_seed,
                                prompt_index,
                                alpha_index,
                            )
                            if method == "random"
                            else None
                        )

                        # Reset token-sampling RNG to the paired baseline state.
                        set_generation_seed(generation_seed)
                        steering_text, diagnostics = generate_with_method(
                            prompt=prompt,
                            method=method,
                            alpha=float(alpha),
                            gate_seed=method_gate_seed,
                        )

                        record: Dict[str, Any] = {
                            "experiment_split": "test",
                            "method": method,
                            "concept_name": CONCEPT_NAME,
                            "feature_id": int(CONCEPT_FEATURE_ID),
                            "prompt_id": int(prompt_index),
                            "prompt": prompt,
                            "no_steering": no_steering,
                            "steering": steering_text,
                            "alpha": float(alpha),
                            "seed": int(generation_seed),
                            "gate_seed": int(method_gate_seed) if method_gate_seed is not None else None,
                            "random_gate_probability": (
                                float(RANDOM_GATE_PROBABILITY) if method == "random" else None
                            ),
                            "content_gate_threshold": (
                                float(CONTENT_GATE_THRESHOLD) if method == "gate" else None
                            ),
                            "gate_calls": int(diagnostics["gate_calls"]),
                            "gate_triggered": int(diagnostics["gate_triggered"]),
                            "gate_trigger_rate": float(diagnostics["gate_trigger_rate"]),
                            "content_score_min": diagnostics["content_score_min"],
                            "content_score_mean": diagnostics["content_score_mean"],
                            "content_score_max": diagnostics["content_score_max"],
                            "raw_vector_norm": float(raw_vector_norm_value),
                            "normalized_vector_norm": float(normalized_vector_norm_value),
                            "temperature": float(TEMPERATURE),
                            "top_p": float(TOP_P),
                            "top_k": int(TOP_K),
                            "max_new_tokens": int(MAX_NEW_TOKENS),
                        }

                        if float(alpha) == 0.0 and steering_text != no_steering:
                            alpha_zero_mismatches.append(
                                (method, int(prompt_index), int(generation_seed))
                            )
                            warnings.warn(
                                "alpha=0 mismatch: "
                                f"method={method}, prompt_id={prompt_index}, seed={generation_seed}. "
                                + (
                                    "For random steering, verify that gate_generator remains "
                                    "independent of the global token-sampling RNG."
                                    if method == "random"
                                    else "Check hook isolation and sampling reproducibility."
                                )
                            )

                        writers[method].write(
                            json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n"                        )
                        writers[method].flush()
finally:
    for writer in writers.values():
        writer.close()
    model.reset_hooks()

total_generation_seconds = float(time.perf_counter() - experiment_started_at)
print(f"Generation finished in {total_generation_seconds:.2f} seconds.")
print(f"Expected records per method: {expected_records_per_method}")
if alpha_zero_mismatches:
    print(f"WARNING: {len(alpha_zero_mismatches)} alpha=0 mismatch(es) detected.")
else:
    print("All alpha=0 paired generations match no-steering outputs.")

## 10. Saved-results validation

The JSONL files are re-read and checked for schema consistency, exact counts, unique
experiment keys, method/folder agreement, valid norms and trigger rates, equal experiment
grids, shared paired baselines, and exact alpha-zero equality.

In [ ]:
REQUIRED_FIELDS: Set[str] = {
    "experiment_split", "method", "concept_name", "feature_id", "prompt_id",
    "prompt", "no_steering", "steering", "alpha", "seed", "gate_seed",
    "random_gate_probability", "content_gate_threshold", "gate_calls",
    "gate_triggered", "gate_trigger_rate", "content_score_min",
    "content_score_mean", "content_score_max", "raw_vector_norm",
    "normalized_vector_norm", "temperature", "top_p", "top_k",
    "max_new_tokens",
}


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing result file: {path}")
    records: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON at {path}:{line_number}") from error
    return records


records_by_method: Dict[str, List[Dict[str, Any]]] = {
    method: read_jsonl(path)
    for method, path in output_paths.items()
}

experiment_grids: Dict[str, Set[Tuple[int, float, int]]] = {}
baseline_by_pair: Dict[Tuple[int, int], str] = {}

for method, records in records_by_method.items():
    assert len(records) == expected_records_per_method, (
        f"{method}: expected {expected_records_per_method} records, found {len(records)}"
    )

    unique_keys: Set[Tuple[str, str, int, int, float, int]] = set()
    grid: Set[Tuple[int, float, int]] = set()
    for record_index, record in enumerate(records):
        missing_fields = REQUIRED_FIELDS - set(record)
        assert not missing_fields, (
            f"{method} record {record_index} misses fields: {sorted(missing_fields)}"
        )
        assert record["method"] == method
        assert record["experiment_split"] == "test"
        assert np.isclose(float(record["normalized_vector_norm"]), 1.0, atol=1e-5)
        assert 0.0 <= float(record["gate_trigger_rate"]) <= 1.0
        assert 0 <= int(record["gate_triggered"]) <= int(record["gate_calls"])

        key = (
            record["method"], record["concept_name"], int(record["feature_id"]),
            int(record["prompt_id"]), float(record["alpha"]), int(record["seed"]),
        )
        assert key not in unique_keys, f"Duplicate experimental key: {key}"
        unique_keys.add(key)

        grid_item = (
            int(record["prompt_id"]), float(record["alpha"]), int(record["seed"])
        )
        grid.add(grid_item)

        baseline_key = (int(record["prompt_id"]), int(record["seed"]))
        previous_baseline = baseline_by_pair.setdefault(
            baseline_key,
            record["no_steering"],
        )
        assert previous_baseline == record["no_steering"], (
            f"Inconsistent no_steering for prompt_id={baseline_key[0]}, "
            f"seed={baseline_key[1]}"
        )

        if float(record["alpha"]) == 0.0:
            assert record["steering"] == record["no_steering"], (
                f"alpha=0 mismatch after reload: method={method}, "
                f"prompt_id={record['prompt_id']}, seed={record['seed']}"
            )

        if method == "random":
            assert record["gate_seed"] is not None
            assert record["random_gate_probability"] is not None
        else:
            assert record["gate_seed"] is None
            assert record["random_gate_probability"] is None

        if method == "gate":
            assert record["content_gate_threshold"] is not None
            assert record["content_score_min"] is not None
            assert record["content_score_mean"] is not None
            assert record["content_score_max"] is not None
        else:
            assert record["content_gate_threshold"] is None
            assert record["content_score_min"] is None
            assert record["content_score_mean"] is None
            assert record["content_score_max"] is None

    experiment_grids[method] = grid

reference_grid = experiment_grids[METHODS[0]]
for method in METHODS[1:]:
    assert experiment_grids[method] == reference_grid, (
        f"Experiment grid mismatch for method {method}"
    )

summary_rows = []
for method in METHODS:
    trigger_rates = np.asarray(
        [float(record["gate_trigger_rate"]) for record in records_by_method[method]],
        dtype=np.float64,
    )
    summary_rows.append({
        "method": method,
        "records": int(len(records_by_method[method])),
        "mean trigger rate": float(trigger_rates.mean()),
        "min trigger rate": float(trigger_rates.min()),
        "max trigger rate": float(trigger_rates.max()),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
print("Saved-results validation passed.")

## 11. Experiment summary

This final cell reports the fixed experiment dimensions, vector norms, runtime, and result
locations. The JSONL files are ready for a separate scoring and Pareto-analysis stage.

In [ ]:
print(f"Prompts: {len(test_prompts)}")
print(f"Seeds: {len(SEEDS)}")
print(f"Alpha values: {len(ALPHAS)}")
print(f"Feature ID: {CONCEPT_FEATURE_ID}")
print(f"Raw vector norm: {raw_vector_norm_value:.8f}")
print(f"Normalized vector norm: {normalized_vector_norm_value:.8f}")
print(f"Total generation time: {total_generation_seconds:.2f} seconds")
print("Saved files:")
for method, path in output_paths.items():
    print(f"  {method}: {path}")